In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# Setup
import os, sys, json, random, csv, zipfile
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import (efficientnet_b0, EfficientNet_B0_Weights,
                                 vit_b_16, ViT_B_16_Weights)
from PIL import Image
from sklearn.metrics import (accuracy_score, f1_score,
                              confusion_matrix, cohen_kappa_score)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

os.system('pip install -q timm pytorch-ignite')
import timm
from ignite.engine import Engine, Events
from ignite.metrics import Accuracy, Loss, RunningAverage
from ignite.handlers import EarlyStopping

os.chdir('/kaggle/working')
if not os.path.exists('fyp-food-classification'):
    os.system('git clone https://github.com/Ahmad-techs/fyp-food-classification.git')
else:
    os.system('cd fyp-food-classification && git pull')
sys.path.insert(0, '/kaggle/working/fyp-food-classification/src')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

def set_seeds(seed=42):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seeds(42)

FOOD5K_ROOT = '/kaggle/input/datasets/binhminhs10/food5k/Food-5K'

OUT_DIR  = '/kaggle/working/results_food5k_v3'
CKPT_DIR = '/kaggle/working/checkpoints_food5k_v3'
os.makedirs(OUT_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
CLASS_NAMES_5K = ['Non-Food', 'Food']

print('Setup complete')

Cloning into 'fyp-food-classification'...


Device: cuda
GPU: Tesla T4
Setup complete


In [2]:
# Dataset and DataLoaders

for split in ['training', 'validation', 'evaluation']:
    p = os.path.join(FOOD5K_ROOT, split)
    files = [f for f in os.listdir(p) if f.endswith('.jpg')]
    food    = sum(1 for f in files if f.startswith('1_'))
    nonfood = sum(1 for f in files if f.startswith('0_'))
    print(f'{split:12s}: {len(files):5d} images  (food={food}, non_food={nonfood})')


class Food5kDataset(Dataset):
    """Food-5k binary dataset. 1_xxx.jpg = Food (1), 0_xxx.jpg = Non-Food (0).
    Returns (image, fine_label, coarse_label) — coarse == fine for binary task."""

    def __init__(self, root, split):
        self.path   = os.path.join(root, split)
        self.images = [f for f in os.listdir(self.path) if f.lower().endswith('.jpg')]
        if split == 'training':
            self.transform = transforms.Compose([
                transforms.Resize(256),
                transforms.RandomCrop(224),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.2, contrast=0.2,
                                       saturation=0.2, hue=0.05),
                transforms.ToTensor(),
                transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(224),
                transforms.ToTensor(),
                transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
            ])

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        name  = self.images[idx]
        label = 1 if name.startswith('1_') else 0
        try:
            img = Image.open(os.path.join(self.path, name)).convert('RGB')
            return self.transform(img), label, label
        except Exception:
            return self.__getitem__((idx + 1) % len(self.images))

    def filename_at(self, idx):
        return self.images[idx]


train_ds = Food5kDataset(FOOD5K_ROOT, 'training')
val_ds   = Food5kDataset(FOOD5K_ROOT, 'validation')
test_ds  = Food5kDataset(FOOD5K_ROOT, 'evaluation')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

imgs, fl, cl = next(iter(train_loader))
print(f'\nBatch: {imgs.shape}')
print(f'Labels: {sorted(fl.unique().tolist())}  <- must be [0, 1]')
assert set(fl.unique().tolist()).issubset({0, 1}), 'Label error!'
print('DataLoaders ready')


training    :  3000 images  (food=1500, non_food=1500)
validation  :  1000 images  (food=500, non_food=500)
evaluation  :  1000 images  (food=500, non_food=500)

Batch: torch.Size([32, 3, 224, 224])
Labels: [0, 1]  <- must be [0, 1]
DataLoaders ready


In [3]:
# Three Model Definitions (dual head, binary)
class EfficientNetDualHead(nn.Module):
    def __init__(self, num_fine=2, num_coarse=2, dropout=0.3):
        super().__init__()
        b = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.features    = b.features
        self.avgpool     = b.avgpool
        self.dropout     = nn.Dropout(dropout)
        self.fine_head   = nn.Linear(1280, num_fine)
        self.coarse_head = nn.Linear(1280, num_coarse)

    def forward(self, x):
        x = self.features(x); x = self.avgpool(x)
        x = torch.flatten(x, 1); x = self.dropout(x)
        return self.fine_head(x), self.coarse_head(x)

    def freeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad = False
    def unfreeze_top(self, n=3):
        for block in list(self.features.children())[-n:]:
            for p in block.parameters(): p.requires_grad = True
    def unfreeze_all(self):
        for p in self.features.parameters(): p.requires_grad = True


class ViTDualHead(nn.Module):
    def __init__(self, num_fine=2, num_coarse=2, dropout=0.3):
        super().__init__()
        vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        vit.heads     = nn.Identity()
        self.backbone = vit
        self.dropout  = nn.Dropout(dropout)
        self.fine_head   = nn.Linear(768, num_fine)
        self.coarse_head = nn.Linear(768, num_coarse)

    def forward(self, x):
        x = self.backbone(x); x = self.dropout(x)
        return self.fine_head(x), self.coarse_head(x)

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_top(self, n=3):
        layers = list(self.backbone.encoder.layers.children())
        for layer in layers[-n:]:
            for p in layer.parameters(): p.requires_grad = True
        for p in self.backbone.encoder.ln.parameters(): p.requires_grad = True
    def unfreeze_all(self):
        for p in self.backbone.parameters(): p.requires_grad = True


class CoAtNetDualHead(nn.Module):
    def __init__(self, num_fine=2, num_coarse=2, dropout=0.3):
        super().__init__()
        self.backbone    = timm.create_model('coatnet_0_rw_224', pretrained=True, num_classes=0)
        feat_dim         = self.backbone.num_features
        self.dropout     = nn.Dropout(dropout)
        self.fine_head   = nn.Linear(feat_dim, num_fine)
        self.coarse_head = nn.Linear(feat_dim, num_coarse)

    def forward(self, x):
        x = self.backbone(x); x = self.dropout(x)
        return self.fine_head(x), self.coarse_head(x)

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_top(self, n=3):
        for name, mod in list(self.backbone.named_children())[-n:]:
            for p in mod.parameters(): p.requires_grad = True
    def unfreeze_all(self):
        for p in self.backbone.parameters(): p.requires_grad = True


print('Testing model shapes (binary: num_fine=2) ...')
dummy = torch.zeros(2, 3, 224, 224)
with torch.no_grad():
    for Cls, name in [(EfficientNetDualHead, 'EfficientNet-B0'),
                       (ViTDualHead,          'ViT-B/16'),
                       (CoAtNetDualHead,      'CoAtNet-0')]:
        m = Cls()
        f, c = m(dummy)
        p = sum(x.numel() for x in m.parameters()) / 1e6
        assert f.shape == (2, 2) and c.shape == (2, 2)
        print(f'  {name:15s}: fine={f.shape}  coarse={c.shape}  params={p:.1f}M ✓')
        del m
print('All models verified')

Testing model shapes (binary: num_fine=2) ...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 131MB/s] 


  EfficientNet-B0: fine=torch.Size([2, 2])  coarse=torch.Size([2, 2])  params=4.0M ✓
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 195MB/s]  


  ViT-B/16       : fine=torch.Size([2, 2])  coarse=torch.Size([2, 2])  params=85.8M ✓


model.safetensors:   0%|          | 0.00/110M [00:00<?, ?B/s]

  CoAtNet-0      : fine=torch.Size([2, 2])  coarse=torch.Size([2, 2])  params=26.7M ✓
All models verified


In [5]:
#Ignite-based Training (Phase 1: 10 epochs | Phase 2: EarlyStopping patience=5)
LAM = 0.35

def make_loss_fn(lam):
    crit = nn.CrossEntropyLoss()
    def loss_fn(fine_out, coarse_out, fl, cl):
        if lam > 0:
            return (1 - lam) * crit(fine_out, fl) + lam * crit(coarse_out, cl)
        return crit(fine_out, fl)
    return loss_fn


def build_engines(model, lr, lam):
    loss_fn = make_loss_fn(lam)
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad],
                       lr=lr, weight_decay=1e-4)

    def train_step(engine, batch):
        model.train()
        imgs, fl, cl = batch
        imgs, fl, cl = imgs.to(device), fl.to(device), cl.to(device)
        optimizer.zero_grad()
        fine_out, coarse_out = model(imgs)
        loss = loss_fn(fine_out, coarse_out, fl, cl)
        loss.backward()
        optimizer.step()
        return {'loss': loss.item()}

    def eval_step(engine, batch):
        model.eval()
        with torch.no_grad():
            imgs, fl, cl = batch
            imgs, fl, cl = imgs.to(device), fl.to(device), cl.to(device)
            fine_out, coarse_out = model(imgs)
            loss = loss_fn(fine_out, coarse_out, fl, cl)
        return {'loss': loss.item(), 'fine_pred': fine_out, 'fine_y': fl,
                'coarse_pred': coarse_out, 'coarse_y': cl}

    trainer   = Engine(train_step)
    evaluator = Engine(eval_step)

    RunningAverage(output_transform=lambda o: o['loss']).attach(trainer, 'loss')
    Accuracy(output_transform=lambda o: (o['fine_pred'],   o['fine_y'])).attach(evaluator, 'fine_acc')
    Accuracy(output_transform=lambda o: (o['coarse_pred'], o['coarse_y'])).attach(evaluator, 'coarse_acc')
    Loss(nn.CrossEntropyLoss(), output_transform=lambda o: (o['fine_pred'], o['fine_y'])).attach(evaluator, 'val_loss')

    return trainer, evaluator, optimizer


def run_phase(model, phase_tag, max_epochs, lr, lam, save_path,
              history, use_early_stopping, patience=5):
    set_seeds(42)
    trainer, evaluator, _ = build_engines(model, lr=lr, lam=lam)
    best = {'fine_acc': history.get('best_fine_acc', 0.0)}

    @trainer.on(Events.EPOCH_COMPLETED)
    def _run_validation(engine):
        evaluator.run(val_loader)
        m = evaluator.state.metrics
        fa, ca, vl = m['fine_acc'] * 100, m['coarse_acc'] * 100, m['val_loss']
        avg_loss = trainer.state.metrics.get('loss', float('nan'))
        print(f'  {phase_tag} Ep{engine.state.epoch:03d}/{max_epochs}  '
              f'train_loss={avg_loss:.4f}  val_loss={vl:.4f}  '
              f'Fine={fa:.2f}%  Coarse={ca:.2f}%')

        history.setdefault('val_fine',   []).append(fa)
        history.setdefault('val_coarse', []).append(ca)
        history.setdefault('val_loss',   []).append(vl)
        history.setdefault('loss',       []).append(avg_loss)

        if fa > best['fine_acc']:
            best['fine_acc'] = fa
            history['best_fine_acc'] = fa
            torch.save({
                'model_state_dict': model.state_dict(),
                'best_fine_acc':    fa,
                'lambda':           lam,
                'epoch':            engine.state.epoch,
                'phase':            phase_tag,
            }, save_path)
            print(f'  ✓ Checkpoint saved (best Fine={fa:.2f}%)')

    if use_early_stopping:
        def score_function(engine):
            return -engine.state.metrics['val_loss']

        es_handler = EarlyStopping(patience=patience, score_function=score_function, trainer=trainer)
        evaluator.add_event_handler(Events.COMPLETED, es_handler)
        print(f'  EarlyStopping attached: patience={patience}, monitors val_loss (min)')

    trainer.run(train_loader, max_epochs=max_epochs)
    return history


def train_model_v3(ModelClass, model_name, save_path, lam=LAM, phase2_max_epochs=100):
    print(f'\n{"="*65}')
    print(f'  {model_name}  |  lambda={lam}')
    print(f'  Schedule: P1=10ep (heads, no ES) | P2<= {phase2_max_epochs}ep + EarlyStopping(patience=5)')
    print(f'{"="*65}')

    model = ModelClass(num_fine=2, num_coarse=2).to(device)
    h = {}

    print('\n-- Phase 1: Heads only | 10 epochs | lr=1e-3 | no early stop --')
    model.freeze_backbone()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'   Trainable params: {trainable:,}')
    h = run_phase(model, 'P1', max_epochs=10, lr=1e-3, lam=lam,
                  save_path=save_path, history=h, use_early_stopping=False)

    print('\n-- Phase 2: Top-3 unfrozen | EarlyStopping patience=5 | lr=1e-4 --')
    model.unfreeze_top(n=3)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'   Trainable params: {trainable:,}')
    h = run_phase(model, 'P2', max_epochs=phase2_max_epochs, lr=1e-4, lam=lam,
                  save_path=save_path, history=h, use_early_stopping=True, patience=5)

    print(f'\n✓ {model_name} done.')
    print(f'  Total epochs run: {len(h["val_fine"])}')
    print(f'  Best val Fine Top-1: {h["best_fine_acc"]:.2f}%')
    return h

print('Ignite training functions ready')

Ignite training functions ready


In [6]:
#Train EfficientNet-B0
h_eff = train_model_v3(EfficientNetDualHead, 'EfficientNet-B0', f'{CKPT_DIR}/food5k_v3_effnet.pth')
print(f'\n EfficientNet-B0: {h_eff["best_fine_acc"]:.2f}% ({len(h_eff["val_fine"])} epochs)')
with open(f'{OUT_DIR}/effnet_history.json', 'w') as f:
    json.dump({k: v for k, v in h_eff.items() if isinstance(v, list)}, f)


  EfficientNet-B0  |  lambda=0.35
  Schedule: P1=10ep (heads, no ES) | P2<= 100ep + EarlyStopping(patience=5)

-- Phase 1: Heads only | 10 epochs | lr=1e-3 | no early stop --
   Trainable params: 5,124
  P1 Ep001/10  train_loss=0.2958  val_loss=0.1460  Fine=96.10%  Coarse=96.30%
  ✓ Checkpoint saved (best Fine=96.10%)
  P1 Ep002/10  train_loss=0.1588  val_loss=0.1115  Fine=97.00%  Coarse=96.40%
  ✓ Checkpoint saved (best Fine=97.00%)
  P1 Ep003/10  train_loss=0.1196  val_loss=0.0897  Fine=97.40%  Coarse=97.30%
  ✓ Checkpoint saved (best Fine=97.40%)
  P1 Ep004/10  train_loss=0.1100  val_loss=0.0848  Fine=97.50%  Coarse=97.20%
  ✓ Checkpoint saved (best Fine=97.50%)
  P1 Ep005/10  train_loss=0.1031  val_loss=0.0757  Fine=97.80%  Coarse=97.80%
  ✓ Checkpoint saved (best Fine=97.80%)
  P1 Ep006/10  train_loss=0.0930  val_loss=0.0812  Fine=97.20%  Coarse=97.10%
  P1 Ep007/10  train_loss=0.0946  val_loss=0.0701  Fine=97.90%  Coarse=98.00%
  ✓ Checkpoint saved (best Fine=97.90%)
  P1 Ep008/

2026-07-22 21:59:55,309 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


  P2 Ep009/100  train_loss=0.0078  val_loss=0.0372  Fine=98.80%  Coarse=98.80%

✓ EfficientNet-B0 done.
  Total epochs run: 19
  Best val Fine Top-1: 99.00%

 EfficientNet-B0: 99.00% (19 epochs)


In [7]:
h_vit = train_model_v3(ViTDualHead, 'ViT-B/16', f'{CKPT_DIR}/food5k_v3_vit.pth')
print(f'\n ViT-B/16: {h_vit["best_fine_acc"]:.2f}% ({len(h_vit["val_fine"])} epochs)')
with open(f'{OUT_DIR}/vit_history.json', 'w') as f:
    json.dump({k: v for k, v in h_vit.items() if isinstance(v, list)}, f)


  ViT-B/16  |  lambda=0.35
  Schedule: P1=10ep (heads, no ES) | P2<= 100ep + EarlyStopping(patience=5)

-- Phase 1: Heads only | 10 epochs | lr=1e-3 | no early stop --
   Trainable params: 3,076
  P1 Ep001/10  train_loss=0.2871  val_loss=0.1206  Fine=97.10%  Coarse=97.00%
  ✓ Checkpoint saved (best Fine=97.10%)
  P1 Ep002/10  train_loss=0.1273  val_loss=0.0852  Fine=97.60%  Coarse=97.80%
  ✓ Checkpoint saved (best Fine=97.60%)
  P1 Ep003/10  train_loss=0.0787  val_loss=0.0690  Fine=98.10%  Coarse=98.30%
  ✓ Checkpoint saved (best Fine=98.10%)
  P1 Ep004/10  train_loss=0.0613  val_loss=0.0589  Fine=98.80%  Coarse=98.60%
  ✓ Checkpoint saved (best Fine=98.80%)
  P1 Ep005/10  train_loss=0.0536  val_loss=0.0536  Fine=98.70%  Coarse=98.70%
  P1 Ep006/10  train_loss=0.0423  val_loss=0.0502  Fine=98.90%  Coarse=99.00%
  ✓ Checkpoint saved (best Fine=98.90%)
  P1 Ep007/10  train_loss=0.0406  val_loss=0.0472  Fine=99.10%  Coarse=98.90%
  ✓ Checkpoint saved (best Fine=99.10%)
  P1 Ep008/10  tra

2026-07-22 22:27:46,075 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


  P2 Ep006/100  train_loss=0.0073  val_loss=0.0571  Fine=99.00%  Coarse=99.00%

✓ ViT-B/16 done.
  Total epochs run: 16
  Best val Fine Top-1: 99.40%

 ViT-B/16: 99.40% (16 epochs)


In [8]:
h_coat = train_model_v3(CoAtNetDualHead, 'CoAtNet-0', f'{CKPT_DIR}/food5k_v3_coatnet.pth')
print(f'\n CoAtNet-0: {h_coat["best_fine_acc"]:.2f}% ({len(h_coat["val_fine"])} epochs)')
with open(f'{OUT_DIR}/coatnet_history.json', 'w') as f:
    json.dump({k: v for k, v in h_coat.items() if isinstance(v, list)}, f)

histories = {'EfficientNet-B0': h_eff, 'ViT-B/16': h_vit, 'CoAtNet-0': h_coat}
print('\n' + '='*50)
print('FOOD-5K V3 VAL SUMMARY')
print('='*50)
for name, h in histories.items():
    print(f'  {name:18s}: {h["best_fine_acc"]:.2f}%  ({len(h["val_fine"])} epochs)')


  CoAtNet-0  |  lambda=0.35
  Schedule: P1=10ep (heads, no ES) | P2<= 100ep + EarlyStopping(patience=5)

-- Phase 1: Heads only | 10 epochs | lr=1e-3 | no early stop --
   Trainable params: 3,076
  P1 Ep001/10  train_loss=0.2670  val_loss=0.1078  Fine=98.00%  Coarse=97.90%
  ✓ Checkpoint saved (best Fine=98.00%)
  P1 Ep002/10  train_loss=0.1049  val_loss=0.0746  Fine=98.30%  Coarse=98.60%
  ✓ Checkpoint saved (best Fine=98.30%)
  P1 Ep003/10  train_loss=0.0709  val_loss=0.0616  Fine=98.60%  Coarse=98.70%
  ✓ Checkpoint saved (best Fine=98.60%)
  P1 Ep004/10  train_loss=0.0571  val_loss=0.0548  Fine=98.40%  Coarse=98.70%
  P1 Ep005/10  train_loss=0.0468  val_loss=0.0492  Fine=98.50%  Coarse=98.60%
  P1 Ep006/10  train_loss=0.0435  val_loss=0.0450  Fine=98.60%  Coarse=98.70%
  P1 Ep007/10  train_loss=0.0359  val_loss=0.0418  Fine=98.90%  Coarse=98.90%
  ✓ Checkpoint saved (best Fine=98.90%)
  P1 Ep008/10  train_loss=0.0317  val_loss=0.0390  Fine=98.90%  Coarse=98.90%
  P1 Ep009/10  trai

2026-07-22 22:53:16,843 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


  P2 Ep010/100  train_loss=0.0191  val_loss=0.0506  Fine=98.30%  Coarse=98.40%

✓ CoAtNet-0 done.
  Total epochs run: 20
  Best val Fine Top-1: 98.90%

 CoAtNet-0: 98.90% (20 epochs)

FOOD-5K V3 VAL SUMMARY
  EfficientNet-B0   : 99.00%  (19 epochs)
  ViT-B/16          : 99.40%  (16 epochs)
  CoAtNet-0         : 98.90%  (20 epochs)


In [9]:
#  Evaluate on Test Set + Confusion Matrices + Per-sample Prediction Log
# The per-sample CSV (filename, true, pred, confidence) is what Day 2's
# error-analysis script reads to isolate Food->Non-Food misclassifications.

def full_evaluate(model, loader, dataset, class_names, label, out_dir):
    model.eval()
    all_ft, all_fp, all_files, all_conf = [], [], [], []
    idx = 0
    with torch.no_grad():
        for imgs, fl, _ in loader:
            fine_out, _ = model(imgs.to(device))
            probs = torch.softmax(fine_out, dim=1)
            preds = fine_out.argmax(1).cpu().numpy()
            confs = probs.max(1).values.cpu().numpy()
            bs = imgs.size(0)
            for b in range(bs):
                all_files.append(dataset.filename_at(idx))
                idx += 1
            all_fp.extend(preds)
            all_ft.extend(fl.numpy())
            all_conf.extend(confs)

    all_ft, all_fp = np.array(all_ft), np.array(all_fp)

    acc   = accuracy_score(all_ft, all_fp) * 100
    f1    = f1_score(all_ft, all_fp, average='macro', zero_division=0) * 100
    kappa = cohen_kappa_score(all_ft, all_fp)

    print(f'\n{"="*50}\n  {label}  [TEST SET]\n{"="*50}')
    print(f'  Top-1 Accuracy:  {acc:.2f}%')
    print(f'  Macro F1:        {f1:.2f}%')
    print(f'  Cohen Kappa:     {kappa:.4f}')

    cm = confusion_matrix(all_ft, all_fp)
    print('\n  Per-class recall:')
    for i, name in enumerate(class_names):
        if cm[i].sum() > 0:
            print(f'    {name:12s}: {cm[i, i] / cm[i].sum() * 100:.1f}%')

    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=1, cbar_kws={'label': 'Recall (%)'})
    ax.set_xlabel('Predicted Class'); ax.set_ylabel('True Class')
    ax.set_title(f'Confusion Matrix — {label}\n(% of true class)', fontweight='bold')
    plt.tight_layout()
    safe = label.replace(' ', '_').replace('/', '_')
    cm_path = f'{out_dir}/{safe}_cm.png'
    plt.savefig(cm_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  Confusion matrix saved: {cm_path}')

    # Per-sample prediction log — needed for Day 2 error analysis
    pred_log_path = f'{out_dir}/{safe}_test_predictions.csv'
    with open(pred_log_path, 'w', newline='') as fcsv:
        writer = csv.writer(fcsv)
        writer.writerow(['filename', 'true_label', 'pred_label', 'confidence'])
        for fn, t, p, c in zip(all_files, all_ft, all_fp, all_conf):
            writer.writerow([fn, int(t), int(p), round(float(c), 4)])
    print(f'  Per-sample prediction log saved: {pred_log_path}')

    return {'condition': label, 'top1': acc, 'f1': f1, 'kappa': kappa, 'cm': cm,
            'pred_log_path': pred_log_path}


experiments = [
    ('EfficientNet-B0', EfficientNetDualHead, f'{CKPT_DIR}/food5k_v3_effnet.pth'),
    ('ViT-B/16',        ViTDualHead,          f'{CKPT_DIR}/food5k_v3_vit.pth'),
    ('CoAtNet-0',       CoAtNetDualHead,      f'{CKPT_DIR}/food5k_v3_coatnet.pth'),
]

results = []
for label, ModelClass, ckpt_path in experiments:
    ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = ModelClass(num_fine=2, num_coarse=2).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    r = full_evaluate(model, test_loader, test_ds, CLASS_NAMES_5K, f'{label} Food-5k', OUT_DIR)
    results.append(r)

print('\n' + '='*55)
print('FOOD-5K FINAL TEST RESULTS')
print('='*55)
print(f'{"Model":20s} {"Top-1":>8} {"F1":>8} {"Kappa":>8}')
print('-'*55)
for r in results:
    print(f'{r["condition"]:20s} {r["top1"]:>8.2f}% {r["f1"]:>7.2f}% {r["kappa"]:>8.4f}')


  EfficientNet-B0 Food-5k  [TEST SET]
  Top-1 Accuracy:  98.80%
  Macro F1:        98.80%
  Cohen Kappa:     0.9760

  Per-class recall:
    Non-Food    : 98.2%
    Food        : 99.4%
  Confusion matrix saved: /kaggle/working/results_food5k_v3/EfficientNet-B0_Food-5k_cm.png
  Per-sample prediction log saved: /kaggle/working/results_food5k_v3/EfficientNet-B0_Food-5k_test_predictions.csv

  ViT-B/16 Food-5k  [TEST SET]
  Top-1 Accuracy:  99.40%
  Macro F1:        99.40%
  Cohen Kappa:     0.9880

  Per-class recall:
    Non-Food    : 99.8%
    Food        : 99.0%
  Confusion matrix saved: /kaggle/working/results_food5k_v3/ViT-B_16_Food-5k_cm.png
  Per-sample prediction log saved: /kaggle/working/results_food5k_v3/ViT-B_16_Food-5k_test_predictions.csv

  CoAtNet-0 Food-5k  [TEST SET]
  Top-1 Accuracy:  99.10%
  Macro F1:        99.10%
  Cohen Kappa:     0.9820

  Per-class recall:
    Non-Food    : 98.6%
    Food        : 99.6%
  Confusion matrix saved: /kaggle/working/results_food5k_v3

In [12]:
print('EPOCH COUNT PER MODEL (for paper methodology section)')
print('='*55)
print(f'{"Model":20s} {"P1":>6} {"P2":>6} {"Total":>8}')
print('-'*55)
p1_fixed = 10
epoch_records = {}
for name, h in histories.items():
    total  = len(h['val_fine'])
    p2_run = max(0, total - p1_fixed)
    epoch_records[name] = {'P1': p1_fixed, 'P2': p2_run, 'Total': total}
    print(f'{name:20s} {p1_fixed:>6} {p2_run:>6} {total:>8}')

print('\nNote for paper:')
print('  Phase 1: 10 epochs (heads only, no early stopping)')
print('  Phase 2: EarlyStopping (ignite handler), patience=5, monitors val_loss')
print(f'  lambda (coarse loss weight): {LAM}')
print('  Optimizer: AdamW (weight decay=1e-4)')

with open(f'{OUT_DIR}/epoch_counts.json', 'w') as f:
    json.dump(epoch_records, f, indent=2)
print(f'\nEpoch counts saved to epoch_counts.json')

EPOCH COUNT PER MODEL (for paper methodology section)
Model                    P1     P2    Total
-------------------------------------------------------
EfficientNet-B0          10      9       19
ViT-B/16                 10      6       16
CoAtNet-0                10     10       20

Note for paper:
  Phase 1: 10 epochs (heads only, no early stopping)
  Phase 2: EarlyStopping (ignite handler), patience=5, monitors val_loss
  lambda (coarse loss weight): 0.35
  Optimizer: AdamW (weight decay=1e-4)

Epoch counts saved to epoch_counts.json


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Food-5k (v3) — Training Curves\nPhase 1: 10ep | Phase 2: EarlyStopping(patience=5)',
             fontsize=12, fontweight='bold')
colours = ['#c0392b', '#2980b9', '#8e44ad']

for (name, h), col in zip(histories.items(), colours):
    axes[0].plot(h['val_loss'], label=name, color=col)
    axes[1].plot(h['val_fine'], label=name, color=col)

for ax, title, ylabel in [(axes[0], 'Validation Loss', 'Loss'),
                          (axes[1], 'Val Fine Top-1 (%)', 'Accuracy (%)')]:
    ax.set_title(title, fontsize=11); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/food5k_v3_training_curves.png', dpi=300, bbox_inches='tight')
plt.close()
print('Training curves saved.')

fig, ax = plt.subplots(figsize=(8, 5))
models = [r['condition'] for r in results]
accs   = [r['top1']      for r in results]
bars   = ax.bar(models, accs, color=colours, edgecolor='white', width=0.5)
ax.set_title('Food-5k Test Accuracy — 3 Models (Updated Schedule)', fontsize=12, fontweight='bold')
ax.set_ylabel('Top-1 Accuracy (%)')
ax.set_ylim(95, 100)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
for bar, v in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{v:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/food5k_v3_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print('Comparison chart saved.')

Training curves saved.
Comparison chart saved.


In [14]:
csv_path = f'{OUT_DIR}/FOOD5K_V3_RESULTS.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['condition', 'top1', 'f1', 'kappa'])
    writer.writeheader()
    for r in results:
        writer.writerow({'condition': r['condition'], 'top1': round(r['top1'], 2),
                          'f1': round(r['f1'], 2), 'kappa': round(r['kappa'], 4)})
print(f'CSV saved: {csv_path}')

zip_path = '/kaggle/working/food5k_v3_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in [OUT_DIR, CKPT_DIR]:
        for root, dirs, files in os.walk(folder):
            for file in files:
                fp   = os.path.join(root, file)
                name = os.path.relpath(fp, '/kaggle/working')
                zf.write(fp, name)
print('Zip ready.')

from IPython.display import FileLink, display
display(FileLink('food5k_v3_results.zip'))


CSV saved: /kaggle/working/results_food5k_v3/FOOD5K_V3_RESULTS.csv
Zip ready.


/kaggle/working/food5k_v3_results.zip

In [18]:
import math
import pandas as pd

ERR_DIR = f'{OUT_DIR}/error_analysis'
os.makedirs(ERR_DIR, exist_ok=True)

summary_rows, all_food_as_nonfood = [], {}
for r in results:
    model_name = r['condition'].replace(' Food-5k', '')
    df = pd.read_csv(r['pred_log_path'])
    total = len(df)

    food_as_nonfood = df[(df['true_label'] == 1) & (df['pred_label'] == 0)].sort_values('confidence', ascending=False)
    nonfood_as_food = df[(df['true_label'] == 0) & (df['pred_label'] == 1)].sort_values('confidence', ascending=False)

    safe = model_name.replace(' ', '_').replace('/', '_')
    food_as_nonfood.to_csv(f'{ERR_DIR}/{safe}_food_as_nonfood_errors.csv', index=False)
    nonfood_as_food.to_csv(f'{ERR_DIR}/{safe}_nonfood_as_food_errors.csv', index=False)
    all_food_as_nonfood[model_name] = food_as_nonfood

    print(f'{model_name}: Food->Non-Food={len(food_as_nonfood)} ({100*len(food_as_nonfood)/total:.2f}%)  '
          f'Non-Food->Food={len(nonfood_as_food)} ({100*len(nonfood_as_food)/total:.2f}%)')

    summary_rows.append({'model': model_name, 'total': total,
                          'food_as_nonfood': len(food_as_nonfood), 'nonfood_as_food': len(nonfood_as_food)})

pd.DataFrame(summary_rows).to_csv(f'{ERR_DIR}/error_analysis_summary.csv', index=False)
print(f'Summary saved: {ERR_DIR}/error_analysis_summary.csv')

EfficientNet-B0: Food->Non-Food=3 (0.30%)  Non-Food->Food=9 (0.90%)
ViT-B/16: Food->Non-Food=5 (0.50%)  Non-Food->Food=1 (0.10%)
CoAtNet-0: Food->Non-Food=2 (0.20%)  Non-Food->Food=7 (0.70%)
Summary saved: /kaggle/working/results_food5k_v3/error_analysis/error_analysis_summary.csv


In [19]:
# %% Cell 13 — Contact sheets of Food->Non-Food errors
def build_contact_sheet(df, model_name, max_images=24, cols=6):
    if len(df) == 0:
        print(f'{model_name}: no Food->Non-Food errors.'); return
    rows_n = min(len(df), max_images); ncols = min(cols, rows_n); nrows = math.ceil(rows_n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3*nrows))
    axes = np.array(axes).reshape(-1)
    for i in range(nrows * ncols):
        ax = axes[i]; ax.axis('off')
        if i < rows_n:
            fname, conf = df.iloc[i]['filename'], df.iloc[i]['confidence']
            img_path = os.path.join(FOOD5K_ROOT, 'evaluation', fname)
            if os.path.exists(img_path):
                ax.imshow(Image.open(img_path).convert('RGB'))
            ax.set_title(f'{fname}\nconf={conf:.2f}', fontsize=7)
    fig.suptitle(f'{model_name} — Food misclassified as Non-Food (showing {rows_n} of {len(df)})',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    safe = model_name.replace(' ', '_').replace('/', '_')
    out_path = f'{ERR_DIR}/{safe}_food_as_nonfood_contact_sheet.png'
    plt.savefig(out_path, dpi=200, bbox_inches='tight'); plt.close()
    print(f'{model_name}: contact sheet saved -> {out_path}')

for model_name, df in all_food_as_nonfood.items():
    build_contact_sheet(df, model_name)

print('\nOpen each *_food_as_nonfood_contact_sheet.png and *_food_as_nonfood_errors.csv')
print('to document recurring visual patterns in the results section.')

EfficientNet-B0: contact sheet saved -> /kaggle/working/results_food5k_v3/error_analysis/EfficientNet-B0_food_as_nonfood_contact_sheet.png
ViT-B/16: contact sheet saved -> /kaggle/working/results_food5k_v3/error_analysis/ViT-B_16_food_as_nonfood_contact_sheet.png
CoAtNet-0: contact sheet saved -> /kaggle/working/results_food5k_v3/error_analysis/CoAtNet-0_food_as_nonfood_contact_sheet.png

Open each *_food_as_nonfood_contact_sheet.png and *_food_as_nonfood_errors.csv
to document recurring visual patterns in the results section.


In [20]:
zip_path = '/kaggle/working/food5k_v3_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in [OUT_DIR, CKPT_DIR]:
        for root, dirs, files in os.walk(folder):
            for file in files:
                fp = os.path.join(root, file)
                zf.write(fp, os.path.relpath(fp, '/kaggle/working'))
print('Zip re-created with error analysis included.')

from IPython.display import FileLink, display
display(FileLink('food5k_v3_results.zip'))

Zip re-created with error analysis included.


/kaggle/working/food5k_v3_results.zip